In [ ]:
# Cell 0 — Kaggle pip installs, path constants, feature flags.
# Runs top-to-bottom. Non-Kaggle (local Windows) path short-circuits installs.

import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    # Pinned, fail-loud installs. bm25s + CharSplit + nltk are small; faiss-gpu is best-effort.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "bm25s>=0.2.0", "CharSplit", "nltk"])
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "faiss-gpu-cu12>=1.14"])
        USE_FAISS_GPU = True
    except subprocess.CalledProcessError:
        print("faiss-gpu-cu12 install failed; falling back to faiss-cpu", flush=True)
        USE_FAISS_GPU = False
    import nltk
    nltk.download("stopwords", quiet=True)
else:
    USE_FAISS_GPU = False  # local Windows dev always CPU faiss

# === Path constants (extends solution.py:20-35 with BGE_M3_DIR + OPUS_MT_DIR) ===
if IS_KAGGLE:
    DATA_DIR    = Path("/kaggle/input/llm-agentic-legal-information-retrieval")
    BGE_M3_DIR  = Path("/kaggle/input/bge-m3")
    OPUS_MT_DIR = Path("/kaggle/input/opus-mt-tc-big-en-de")
    RERANK_DIR  = Path("/kaggle/input/mmarco-reranker")
    OUT_PATH    = Path("/kaggle/working/submission.csv")
else:
    _ROOT = Path(r"C:\Users\Dharun prasanth\OneDrive\Documents\Projects\LLm_Agentic")
    DATA_DIR    = _ROOT / "Data"
    BGE_M3_DIR  = _ROOT / "models" / "bge-m3"
    OPUS_MT_DIR = _ROOT / "models" / "opus-mt-tc-big-en-de"
    RERANK_DIR  = _ROOT / "models" / "mmarco-reranker"
    OUT_PATH    = _ROOT / "submission.csv"

# === Feature flags (INC-02) ===
USE_RERANKER           = True    # Phase 1: on
USE_COURT_CORPUS       = False   # Phase 2+ enables
USE_ENTITY_GRAPH       = False   # Phase 3+ enables
USE_COURT_DENSE_RERANK = False   # Phase 4+ enables
USE_JINA_CROSS_ENCODER = False   # Phase 4+ replaces mmarco
USE_LLM_AUGMENTATION   = False   # Phase 5+ enables
SMOKE                  = os.environ.get("SMOKE", "0") == "1"

# === Hyperparameters (tune on val only) ===
BM25_LAWS_K   = 100
DENSE_LAWS_K  = 100
RRF_K_CONST   = 60
RERANK_K      = 150
SMOKE_LAWS_N  = 5000
SMOKE_VAL_N   = 3
SEED          = 42

print(f"IS_KAGGLE={IS_KAGGLE}  USE_FAISS_GPU={USE_FAISS_GPU}  SMOKE={SMOKE}")


In [ ]:
# Cell 1 — HARD GATE: abort if GPU is not present. AH-1 prevention.
import torch, time, gc, re, random
import numpy as np
import pandas as pd

assert torch.cuda.is_available(), (
    "GPU unavailable. Check Kaggle Settings -> Accelerator -> GPU T4 x2. "
    "AH-1 prevention: this notebook refuses to run on CPU."
)

DEVICE = torch.device("cuda")
_dev_name = torch.cuda.get_device_name(0)
_vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
_vram_free  = torch.cuda.mem_get_info()[0] / 1e9
print(f"Device: {_dev_name}")
print(f"VRAM total: {_vram_total:.1f} GB")
print(f"VRAM free:  {_vram_free:.1f} GB")

# Deterministic seeds
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Small helper reused by every model stage (D-06)
def unload(*objs):
    """Free GPU memory: del refs, gc collect, empty_cache, log VRAM free."""
    for _o in objs:
        del _o
    gc.collect()
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f"  unloaded -> VRAM free: {_free/1e9:.1f}/{_total/1e9:.1f} GB", flush=True)


In [ ]:
# Cell 2 — Load laws/val/test CSVs, log shapes, canonicalize laws.

assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"
t0 = time.time()

laws = pd.read_csv(DATA_DIR / "laws_de.csv").fillna("")
val  = pd.read_csv(DATA_DIR / "val.csv").fillna("")
test = pd.read_csv(DATA_DIR / "test.csv").fillna("")

print(f"Loaded CSVs in {time.time()-t0:.1f}s")
print(f"  laws : {laws.shape}  columns={list(laws.columns)}  "
      f"mem={laws.memory_usage(deep=True).sum()/1e6:.0f} MB")
print(f"  val  : {val.shape}   columns={list(val.columns)}")
print(f"  test : {test.shape}  columns={list(test.columns)}")

# A1-A10 data inspection banner for the Kaggle reviewer
print("\n=== Corpus inspection ===")
print(f"laws['citation'] unique count: {laws['citation'].nunique():,}")
print(f"laws['citation'] sample: {laws['citation'].head(5).tolist()}")
print(f"val query count: {len(val)}")
print(f"test query count: {len(test)}")


In [ ]:
# Cell 3 — Canonicalization and tokenization helpers.
# canonicalize() is the single source of truth for query, corpus, and submission
# citation formatting (D-05, LAWS-05, QUERY-04, CALIB-02).

LAW_CODE_ALIASES = {
    "CC":  "ZGB",   # Code civil -> Zivilgesetzbuch
    "CO":  "OR",    # Code des obligations -> Obligationenrecht
    "CP":  "StGB",  # Code penal -> Strafgesetzbuch
    "CPP": "StPO",  # Code de procedure penale -> Strafprozessordnung
    "LP":  "SchKG", # Loi sur la poursuite -> SchKG
    "LTF": "BGG",   # Loi sur le Tribunal federal -> BGG
}

def canonicalize(s):
    """
    Canonicalize to Swiss German canonical form used in laws_de.csv.
    Idempotent. Rules (from 01-RESEARCH.md section Swiss Citation Canonicalization):
      1. Strip + collapse whitespace to single space.
      2. sharp-s (U+00DF) -> 'ss' (Swiss German standard).
      3. 'Art.X' / 'Art.  X' -> 'Art. X' (single space). Same for Abs./lit./Ziff.
      4. Map French-language law code aliases to German (CC->ZGB etc.).
      5. Preserve BGE Roman numerals and 'E.' vs 'E ' variants as-is.
    """
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = s.replace("\u00df", "ss")  # sharp-s -> ss
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\b(Art|Abs|lit|Ziff)\.\s*(\d)", r"\1. \2", s)
    for _fr, _de in LAW_CODE_ALIASES.items():
        s = re.sub(rf"\b{_fr}\b", _de, s)
    return s

# Idempotence smoke check (fail-loud)
assert canonicalize("Art.11 Abs.2 CC") == "Art. 11 Abs. 2 ZGB", \
    f"canonicalize broken: {canonicalize('Art.11 Abs.2 CC')!r}"
assert canonicalize(canonicalize("Art.11 Abs.  2 CC")) == canonicalize("Art.11 Abs.  2 CC")

# --- extract_legal_codes: reused from solution.py:319-330 for D-03 ---
QUERY_CODES = {
    "ZGB", "OR", "StGB", "StPO", "BGG", "SchKG", "BGB",
    "AHVG", "IVG", "ELG", "KVG", "UVG", "BVG",
    "DBG", "MWSTG", "USG", "RPG",
}

def extract_legal_codes(text):
    """
    Pull '(law_code, article_number)' tuples from the ORIGINAL English query
    text for D-03. Returns a space-joined string of legal code tokens to
    append to the German translation before BM25 tokenization.
    """
    if not isinstance(text, str):
        return ""
    out = []
    # Art. N CODE patterns (e.g., "Art. 41 OR")
    for m in re.finditer(r"\bArt\.\s*(\d+[a-z]?)\s*([A-Z][A-Za-z]{1,6})\b", text):
        num, code = m.group(1), m.group(2)
        if code in QUERY_CODES:
            out.append(f"Art. {num} {code}")
    # Bare law code tokens (e.g., "under OR", "ZGB art. 1")
    for m in re.finditer(r"\b([A-Z][A-Za-z]{1,6})\b", text):
        if m.group(1) in QUERY_CODES:
            out.append(m.group(1))
    # BGE references
    for m in re.finditer(r"\bBGE\s+\d+\s+[IVX]+[a-z]?\s+\d+\b", text):
        out.append(m.group(0))
    return " ".join(out)

# --- tokenize_for_bm25_de: legal-aware German tokenizer for bm25s ---
# Full CharSplit decompounding wiring and NLTK stopwords happen here. bm25s
# itself just consumes the list-of-list tokens we produce.
from nltk.corpus import stopwords as _nltk_stopwords
GERMAN_STOP = frozenset(_nltk_stopwords.words("german"))

try:
    from charsplit import Splitter
    _CHARSPLIT = Splitter()
    _CHARSPLIT_OK = True
except Exception as _e:
    print(f"  CharSplit unavailable ({_e}); decompounding disabled.", flush=True)
    _CHARSPLIT = None
    _CHARSPLIT_OK = False

def tokenize_for_bm25_de(text):
    """
    Legal-aware German tokenization for bm25s input.
      1. Lowercase; split on word+dot boundaries (preserves Art., Abs., BGE, E., numbers).
      2. Remove German stopwords (der, die, das, gemaess, ...).
      3. For tokens longer than 3 chars without a period, attempt CharSplit
         decompounding and APPEND (not replace) the head/tail split if score > 0.5.
    Returns list[str].
    """
    text = str(text).lower()
    raw = re.findall(r"[\w.]+", text)
    out = []
    for tok in raw:
        if tok in GERMAN_STOP:
            continue
        if len(tok) <= 3 or "." in tok:
            out.append(tok)
            continue
        out.append(tok)
        if _CHARSPLIT_OK:
            try:
                splits = _CHARSPLIT.split_compound(tok)
                if splits:
                    score, head, tail = splits[0]
                    if score > 0.5 and head and tail:
                        out.append(head.lower())
                        out.append(tail.lower())
            except Exception:
                pass
    return out

# Legal-notation survival smoke check
_sample_tokens = tokenize_for_bm25_de("Die Pflicht aus Art. 41 OR")
assert "art." in _sample_tokens, f"Art. dropped: {_sample_tokens}"
assert "or" in _sample_tokens, f"OR dropped: {_sample_tokens}"

# === LAWS-05: apply canonicalize to laws corpus BEFORE any index build ===
t0 = time.time()
laws["citation"] = laws["citation"].apply(canonicalize)
laws["text"]     = laws["text"].apply(canonicalize)
print(f"Canonicalized laws corpus in {time.time()-t0:.1f}s")

# === Pitfall 6 guard: non-collapsing assertion ===
# canonicalize() must NOT collapse two distinct corpus citations into the same string.
_raw_unique = laws["citation"].drop_duplicates()
assert len(set(canonicalize(c) for c in _raw_unique)) == len(set(_raw_unique)), (
    "canonicalize() collapses distinct corpus citations — shrink LAW_CODE_ALIASES"
)
print(f"Non-collapsing canonicalize assert OK on {len(_raw_unique):,} unique citations")


In [ ]:
# ── Extract legal codes from citation strings ──
# For each law citation, get its 'code' (e.g. 'StPO', 'ZGB')
CODE_PAT = re.compile(r'\b([A-Z][A-Za-z]{1,10})\b\s*$')

def citation_code(cit):
    m = CODE_PAT.search(cit)
    return m.group(1) if m else None

# Precompute code for each law citation (for fast boost later)
laws_codes = np.array([citation_code(c) or '' for c in laws_cits])
print(f'Laws with detected code: {(laws_codes != "").sum()}/{len(laws_cits)}')
print(f'Distinct codes: {len(set(laws_codes))}')
top_codes = Counter(laws_codes).most_common(15)
print(f'Top 15 codes: {top_codes}')

# Query code extraction
QUERY_CODES = {'ZGB','OR','StGB','StPO','ZPO','BGG','SchKG','AHVG','IVG','ATSG',
                'BV','BVG','USG','UVPV','IPRG','SVG','UVG','LAI','DBG','MWStG',
                'URG','UWG','AIG','VStG','VStrR','AHVV','VwVG','RTVG','StBOG',
                'KVG','VZAE','ZStV','AuG','LugÜ','NAG','IRSG','PublG'}
CODE_REGEX = re.compile(r'\b(' + '|'.join(QUERY_CODES) + r')\b')

def extract_query_codes(text):
    return set(CODE_REGEX.findall(text))

# Test on val_001
print(f"\nval_001 extracted codes: {extract_query_codes(val.iloc[0]['query'])}")

In [ ]:
# ── Retrieval function ──
def retrieve(q_en, top_k=500):
    q_emb = encode([q_en], batch_size=1, prefix='query: ', max_len=512)[0]
    scores = laws_embs @ q_emb

    # Boost docs whose code matches query-extracted codes
    query_codes = extract_query_codes(q_en)
    if query_codes:
        code_mask = np.isin(laws_codes, list(query_codes))
        scores[code_mask] += 0.1  # significant boost

    top = np.argsort(-scores)[:top_k]
    return [(laws_cits[i], scores[i]) for i in top]

def predict(q_en, top_k):
    cands = retrieve(q_en, top_k=top_k * 3)
    preds = []
    seen = set()
    # First pass: top scoring
    for cit, score in cands:
        if cit not in seen:
            seen.add(cit)
            preds.append(cit)
            if len(preds) >= top_k:
                break
    # Fallback: add train-frequent citations not already in preds
    for cit in TOP_TRAIN:
        if cit not in seen:
            seen.add(cit)
            preds.append(cit)
            if len(preds) >= top_k + 10:
                break
    return preds[:top_k]

In [ ]:
# ── Validate on val ──
def macro_f1(gold_sets, pred_sets):
    f1s = []
    for g, p in zip(gold_sets, pred_sets):
        if not g and not p: f1s.append(1.0); continue
        if not g or not p:  f1s.append(0.0); continue
        tp = len(g & p)
        pr = tp / len(p)
        rc = tp / len(g)
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(f1s))

print('Processing val...')
val_ranked = []
for i, row in val.iterrows():
    preds = predict(row['query'], top_k=100)
    gold = set(str(row['gold_citations']).split(';'))
    val_ranked.append({'gold': gold, 'ranked': preds})
    tp20 = len(gold & set(preds[:20]))
    tp50 = len(gold & set(preds[:50]))
    print(f'  {row["query_id"]}: gold={len(gold)}  tp@20={tp20}  tp@50={tp50}')

best_f1, best_k = 0, 0
for k in range(1, 101):
    f1 = macro_f1([r['gold'] for r in val_ranked],
                  [set(r['ranked'][:k]) for r in val_ranked])
    if f1 > best_f1: best_f1, best_k = f1, k
print(f'\n*** Val macro-F1 = {best_f1:.4f} @ top-{best_k} ***')

In [ ]:
# ── Generate test predictions ──
print('Processing test...')
rows = []
for i, row in test.iterrows():
    preds = predict(row['query'], top_k=best_k)
    rows.append({'query_id': row['query_id'], 'predicted_citations': ';'.join(preds)})

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv  ({len(submission)} rows)')
submission.head()